In [1]:
import pandas as pd

df = pd.read_csv("../results/qa_uncertainty_signals.csv")

df.head()

,question,predicted,true,correct,start_logits,end_logits,span_prob,start_entropy,end_entropy,confidence_margin,entropy_mean,entropy_diff
0,In what country is Normandy located?,France,France,True,[[-3.02749 -8.174801 -8.407998 -8.194277 ...,[[-2.380471 -8.395869 -7.7617874 -7.94241...,0.999525,0.001243,0.003745,0.999882,0.002494,0.002501
1,When were the Normans in Normandy?,10th and 11th centuries,10th and 11th centuries,True,[[ 0.4090171 -7.866979 -8.080148 -8.300548 ...,[[-0.34856105 -7.8766556 -8.135746 -8.43703...,0.716860,0.790975,0.035410,0.524292,0.413192,0.755565
2,From which countries did the Norse originate?,NaN,"Denmark, Iceland and Norway",False,[[ 6.604309 -8.025348 -8.379556 -8.980244 ...,[[ 6.2046175 -8.419378 -7.6635 -7.2663493 ...,0.416920,0.694885,0.575946,0.030302,0.635416,0.118939
3,Who was the Norse leader?,Rollo,Rollo,True,[[ 6.415034 -7.598161 -8.306515 -7.793644 ...,[[ 4.9600334 -7.9587026 -8.153611 -8.91567...,0.501952,0.690123,0.468602,0.202216,0.579363,0.221521
4,What century did the Normans first gain their ...,10th,10th century,False,[[-0.23108938 -7.7582216 -8.784559 -8.17281...,[[ 0.46058223 -8.097748 -7.632112 -8.20574...,0.361124,1.170431,0.978716,0.594383,1.074573,0.191716


In [2]:
df["correct"].value_counts()

correct
False    1500
True      500
Name: count, dtype: int64

In [3]:
df_true = df[df["correct"] == True]
df_false = df[df["correct"] == False]

# take equal number of False samples
df_false_sample = df_false.sample(len(df_true), random_state=42)

# combine
df_balanced = pd.concat([df_true, df_false_sample])

# shuffle (important)
df_balanced = df_balanced.sample(frac=1, random_state=42)

df_balanced["correct"].value_counts()

correct
False    500
True     500
Name: count, dtype: int64

In [4]:
features = [
    "span_prob",
    "start_entropy",
    "end_entropy",
    "confidence_margin",
    "entropy_mean",
    "entropy_diff"
]

X = df_balanced[features]
y = df_balanced["correct"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

Feature shape: (1000, 6)
Label shape: (1000,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 800
Test size: 200


In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

print("Balanced Random Forest trained")

Balanced Random Forest trained


In [11]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.62

Classification Report:

              precision    recall  f1-score   support

       False       0.63      0.59      0.61       100
        True       0.61      0.65      0.63       100

    accuracy                           0.62       200
   macro avg       0.62      0.62      0.62       200
weighted avg       0.62      0.62      0.62       200



In [12]:
import joblib

joblib.dump(rf_model, "../results/reliability_classifier.pkl")

print("Balanced model saved successfully")

Balanced model saved successfully


In [13]:
import joblib

loaded_model = joblib.load("../results/reliability_classifier.pkl")

print("Model loaded")

Model loaded
